In [0]:
%pip install faker

In [0]:
from faker import Faker
import pandas as pd
import random
from datetime import datetime, timedelta

fake = Faker()

rows = []

for i in range(100_000):
    rows.append({
        "transaction_id": i,
        "user_id": random.randint(1, 5000),
        "amount": round(random.uniform(1, 5000), 2),
        "merchant_category": random.choice(["food", "travel", "shopping", "electronics", "cash"]),
        "country": random.choice(["VN", "US", "JP", "SG"]),
        "transaction_time": fake.date_time_between(
            start_date="-90d",
            end_date="now"
        )
    })

tx_pd = pd.DataFrame(rows)

In [0]:
tx_df = spark.createDataFrame(tx_pd)

tx_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_transactions")

In [0]:
display(spark.table("bronze_transactions"))

In [0]:
tx = spark.table("bronze_transactions")
display(tx)

In [0]:
tx_pd = tx.toPandas()
tx_pd = tx_pd.sort_values(["user_id", "transaction_time"])

In [0]:
tx_pd["hour"] = tx_pd["transaction_time"].dt.hour
tx_pd["day_of_week"] = tx_pd["transaction_time"].dt.dayofweek
tx_pd["is_night"] = tx_pd["hour"].between(0, 5).astype(int)

In [0]:
user_avg = tx_pd.groupby("user_id")["amount"].transform("mean")
user_std = tx_pd.groupby("user_id")["amount"].transform("std")

tx_pd["amount_zscore"] = (tx_pd["amount"] - user_avg) / user_std
tx_pd["amount_zscore"] = tx_pd["amount_zscore"].fillna(0)

In [0]:
tx_pd["tx_count_user"] = tx_pd.groupby("user_id")["transaction_id"].transform("count")

In [0]:
silver_fraud_df = spark.createDataFrame(tx_pd)

silver_fraud_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_fraud_features")

In [0]:
display(spark.table("silver_fraud_features"))